In [1]:
import geopandas as gpd
import pandas as pd
from functools import reduce 
from functions import *

Estimates relative to Residential Housing for the second semester (S2) of 2025 are merged into geographical perimeters.

Load and simplify OMI estimates

In [ ]:
df_omi = pd.read_csv('datasets/omi_estimate/omi_estimate.csv')

# Select 2025_S2 estimate for Residential Housing
df_omi = df_omi[(df_omi['year_semester'] == '2025_S2') & (df_omi['type'] == 'Residential housing')]

add_zeroes(df_omi, 'mun_istat', 6)

,year,year_semester,semester,zone,type,condition,buy_min,buy_max,mun_istat,land_code,mun_name,prov_name,reg_name
0,2004,2004_S1,1,D1,Industrial buildings,Normal,340,490,001001,A074,Agliè,Torino,Piemonte
1,2004,2004_S1,1,D1,Laboratories,Excellent,590,740,001001,A074,Agliè,Torino,Piemonte
2,2004,2004_S1,1,B1,Lowcost housing,Normal,600,750,001001,A074,Agliè,Torino,Piemonte
3,2004,2004_S1,1,C1,Lowcost housing,Normal,600,750,001001,A074,Agliè,Torino,Piemonte
4,2004,2004_S1,1,C2,Lowcost housing,Normal,600,750,001001,A074,Agliè,Torino,Piemonte
...,...,...,...,...,...,...,...,...,...,...,...,...,...
7148755,2025,2025_S2,2,B2,Warehouses,Normal,395,600,078157,M403,Corigliano-Rossano,Cosenza,Calabria
7148756,2025,2025_S2,2,B3,Warehouses,Normal,335,485,078157,M403,Corigliano-Rossano,Cosenza,Calabria
7148757,2025,2025_S2,2,D4,Warehouses,Normal,365,530,078157,M403,Corigliano-Rossano,Cosenza,Calabria
7148758,2025,2025_S2,2,E1,Warehouses,Normal,315,455,078157,M403,Corigliano-Rossano,Cosenza,Calabria


In [3]:
# Groupby [mun_istat, zone, year_semester]
df_omi = df_omi.groupby(['mun_istat', 'zone', 'year_semester']).aggregate({
    'mun_name':'first',
    'prov_name':'first',
    'reg_name':'first',
    'buy_min':'mean',
    'buy_max':'mean'
}).reset_index()

Zone data

In [4]:
# Load zone perimeters
gdf_zone = gpd.read_file('datasets/geo_data/omi_zone_perimeters.gpkg', layer='zones')

In [5]:
# Merge geographical info into estimates
gdf_zone_merged = gpd.GeoDataFrame(pd.merge(
    df_omi, gdf_zone[['mun_istat', 'zone', 'geometry']], 
    on = ['mun_istat', 'zone'], 
    how = 'left'
))

gdf_zone_merged = gdf_zone_merged.dropna(subset = 'geometry')

In [6]:
gdf_zone_merged.to_file('datasets/data_maps/zone_estimate.gpkg')

Municipal data

In [7]:
# Load Municipality perimeters
gdf_mun = gpd.read_file('datasets/geo_data/mun_perimeters.gpkg', layer='mun_perimeters')

In [8]:
# Group by ISTAT code
df_omi_mun = df_omi.groupby('mun_istat').aggregate({
    'mun_name' : 'first',
    'prov_name' : 'first',
    'reg_name' : 'first',
    'year_semester' : 'first',
    'buy_min' : 'mean',
    'buy_max' : 'mean'
}).reset_index()

# Change buy_min/max to int (no decimals)
df_omi_mun[['buy_min', 'buy_max']] = df_omi_mun[['buy_min', 'buy_max']].astype(int)

# Merge geographical info into estimates
gdf_mun_merged = gpd.GeoDataFrame(pd.merge(
    df_omi_mun, gdf_mun[['mun_istat', 'geometry']], 
    on = 'mun_istat', 
    how = 'left'
))

gdf_mun_merged = gdf_mun_merged.dropna()

In [9]:
gdf_mun_merged.to_file('datasets/data_maps/mun_estimate.gpkg')

Province data

In [10]:
# Load Province perimeters
gdf_prov = gpd.read_file('datasets/geo_data/prov_perimeters.gpkg')

In [11]:
# Group by Province names
df_omi_prov = df_omi.groupby('prov_name').aggregate({
    'reg_name' : 'first',
    'year_semester' : 'first',
    'buy_min' : 'mean',
    'buy_max' : 'mean'
}).reset_index()

# Change buy_min/max to int (no decimals)
df_omi_prov[['buy_min', 'buy_max']] = df_omi_prov[['buy_min', 'buy_max']].astype(int)

# Merge geographical info into estimates
gdf_prov_merged = gpd.GeoDataFrame(pd.merge(
    df_omi_prov, gdf_prov[['prov_name', 'geometry']],
    on = 'prov_name',
    how = 'left'
))

In [12]:
gdf_prov_merged.to_file('datasets/data_maps/prov_estimate.gpkg')

Region data

In [13]:
# Load Region perimeters
gdf_reg = gpd.read_file('datasets/geo_data/reg_perimeters.gpkg')

In [14]:
# Group by Region names
df_omi_reg = df_omi.groupby('reg_name').aggregate({
    'year_semester' : 'first',
    'buy_min' : 'mean',
    'buy_max' : 'mean'
}).reset_index()

# Change buy_min/max to int (no decimals)
df_omi_reg[['buy_min', 'buy_max']] = df_omi_reg[['buy_min', 'buy_max']].astype(int)

# Merge geographical info into estimates
gdf_reg_merged = gpd.GeoDataFrame(pd.merge(
    df_omi_reg, gdf_reg,
    on = 'reg_name',
    how = 'left'
))

In [15]:
gdf_reg_merged.to_file('datasets/data_maps/reg_estimate.gpkg')